# 02. M2 전처리 – raw 데이터 6개 점검
각 셀을 위에서부터 `Shift+Enter`로 한 칸씩 실행하세요. 이 노트북은 파일을 **읽기만** 하고 raw는 수정하지 않습니다.

## 1. raw 파일 찾기

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent          # 노트북이 하위 폴더(notebooks/)에 있을 때
RAW = ROOT / "data" / "raw"
files = sorted(p for p in RAW.glob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls"))
print("raw 폴더:", RAW)
for i, p in enumerate(files, 1):
    print(i, p.name)

raw 폴더: /Users/song/Desktop/unit1_project/data/raw
1 1_한국산업안전보건공단_산업재해 고위험요인(SIF) 아카이브_20260401.xlsx
2 2_한국산업안전보건공단_산업중분류별 규모별 사고재해자수_20251231.csv
3 3_한국산업안전보건공단_규모별 사고사망자수_20251231.csv
4 4_한국산업안전보건공단_산업중분류별 규모별 사업장수_20251231.csv
5 5_한국산업안전보건공단_산업중분류별 규모별 사망만인율_20251231.csv
6 6_고용노동부_산업재해 사고정보 예방조치 현황_AI친화 학습데이터_20260917.csv


## 2. CSV: 인코딩 자동 판별 + 크기·컬럼·결측·중복

In [2]:
def read_csv_auto(p):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(p, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {p.name}")

dfs = {}
for p in files:
    if p.suffix.lower() != ".csv":
        continue
    df, enc = read_csv_auto(p)
    df.columns = df.columns.astype(str).str.replace(r"\s+", "", regex=True)
    dfs[p.name] = df
    miss = df.isna().sum()
    print(f"\n== {p.name} | {enc} | {df.shape[0]}행 x {df.shape[1]}열")
    print("컬럼:", list(df.columns))
    print("결측:", miss[miss > 0].to_dict() or "없음", "| 중복 행:", int(df.duplicated().sum()))


== 2_한국산업안전보건공단_산업중분류별 규모별 사고재해자수_20251231.csv | cp949 | 30행 x 12열
컬럼: ['대업종', '구분', '5인미만근로자수', '5인-9인근로자수', '10인-19인근로자수', '20인-29인근로자수', '30인-49인근로자수', '50인-99인근로자수', '100인-299인근로자수', '300인-499인근로자수', '500인-999인근로자수', '1000인이상근로자수']
결측: 없음 | 중복 행: 0

== 3_한국산업안전보건공단_규모별 사고사망자수_20251231.csv | cp949 | 10행 x 23열
컬럼: ['구분', '2004년사고사망자수', '2005년사고사망자수', '2006년사고사망자수', '2007년사고사망자수', '2008년사고사망자수', '2009년사고사망자수', '2010년사고사망자수', '2011년사고사망자수', '2012년사고사망자수', '2013년사고사망자수', '2014년사고사망자수', '2015년사고사망자수', '2016년사고사망자수', '2017년사고사망자수', '2018년사고사망자수', '2019년사고사망자수', '2020년사고사망자수', '2021년사고사망자수', '2022년사고사망자수', '2023년사고사망자수', '2024년사고사망자수', '2025년사고사망자수']
결측: 없음 | 중복 행: 0

== 4_한국산업안전보건공단_산업중분류별 규모별 사업장수_20251231.csv | cp949 | 30행 x 12열
컬럼: ['대업종', '구분', '5인미만사업장수', '5인-9인사업장수', '10인-19인사업장수', '20인-29인사업장수', '30인-49인사업장수', '50인-99인사업장수', '100인-299인사업장수', '300인-499인사업장수', '500인-999인사업장수

## 3. 엑셀: 모든 시트를 열어서 표인지 문서인지 확인
1번 파일은 첫 시트가 비어 있었으므로 시트 전체를 봅니다.

In [3]:
for p in files:
    if p.suffix.lower() not in (".xlsx", ".xls"):
        continue
    sheets = pd.read_excel(p, sheet_name=None, header=None)
    print(f"\n== {p.name} | 시트 {len(sheets)}개")
    for name, df in sheets.items():
        filled = int(df.notna().sum().sum())
        print(f"  [{name}] {df.shape[0]}행 x {df.shape[1]}열, 값 있는 칸 {filled}개")
        for v in df.stack().head(6):
            print("     ·", str(v)[:90].replace("\n", " "))


== 1_한국산업안전보건공단_산업재해 고위험요인(SIF) 아카이브_20260401.xlsx | 시트 3개
  [개요] 0행 x 0열, 값 있는 칸 0개
  [아카이브(제조업등)] 2576행 x 10열, 값 있는 칸 23167개
     · nan
     · □ 제조업 등(건설업 외 업종)
     · nan
     · nan
     · nan
     · nan
  [아카이브(건설업)] 3463행 x 10열, 값 있는 칸 31142개
     · nan
     · □ 건설업
     · nan
     · nan
     · nan
     · nan


## 4. 요약표 (노션에 붙여넣기용)

In [4]:
rows = []
for name, df in dfs.items():
    num = df.select_dtypes("number")
    out = 0
    for c in num.columns:
        q1, q3 = num[c].quantile([0.25, 0.75]); iqr = q3 - q1
        out += int(((num[c] < q1 - 1.5*iqr) | (num[c] > q3 + 1.5*iqr)).sum())
    rows.append([name, df.shape[0], df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum()), out])
summary = pd.DataFrame(rows, columns=["파일", "행", "열", "결측합", "중복", "이상치후보(IQR)"])
summary

,파일,행,열,결측합,중복,이상치후보(IQR)
0,2_한국산업안전보건공단_산업중분류...,30,12,0,0,45
1,3_한국산업안전보건공단_규모별 사고사...,10,23,0,0,18
2,4_한국산업안전보건공단_산업중분류...,30,12,0,0,37
3,5_한국산업안전보건공단_산업중분류...,30,12,14,0,35
4,6_고용노동부_산업재해 사고정보 예방조치 ...,5,4,0,0,0
